# Column with node-to-surface constraints — v5

Same model as v4, but now the static analysis is captured as a **time series** so the viewer shows the load being applied progressively.

## What's new in v5

`ops.integrator('LoadControl', 0.1)` runs the load in **10 pseudo-time steps** (λ = 0.1, 0.2, ..., 1.0). In v4 we only extracted the final step. Here we capture the displacement field after every `ops.analyze(1)` call and pass the whole series to `Results.from_fem(steps=[...])`. The viewer picks this up automatically and enables its time-step slider — drag it to scrub through the loading history, or watch the viewer animate between snapshots.

Each step dict has the shape:

```python
{
    "time": λ,                              # 0.1, 0.2, ..., 1.0
    "point_data": {
        "Displacement": disp_at_step,       # (N, 3)
        "|U|":          u_mag_at_step,      # (N,)
        "Ux":           ux_at_step,         # (N,)
        "Uy":           uy_at_step,         # (N,)
        "Uz":           uz_at_step,         # (N,)
    },
}
```

The scalar range on each contour field auto-adjusts as you scrub, so bending modes that are invisible at step 1 (small load, small displacement) become legible when you zoom the color bar — and symmetrical fields like `Ux` (Poisson contraction) scale linearly with λ as expected.

In [ ]:
from apeGmsh import apeGmsh, Part, Results
from apeGmsh import FEMData
from apeGmsh.sections import W_solid
import numpy as np
from pathlib import Path

In [ ]:
m1 = apeGmsh(model_name='beam_column', verbose=False)
m1.begin()

column = m1.sections.W_solid(bf=150, tf=20, h=300, tw=10, length=2000, label='column')

m1.model.selection.select_volumes().to_physical(name='pg_column')

base = m1.model.geometry.add_point(x=0, y=0, z=0,    lc=100, label='base')
top  = m1.model.geometry.add_point(x=0, y=0, z=2000, lc=100, label='top')

m1.constraints.node_to_surface('base', column.labels.start_face)
m1.constraints.node_to_surface('top',  column.labels.end_face)

m1.loads.point(
    target='top',
    force_xyz=[0, 1000, 0],
    moment_xyz=[0, 0, 0],
)

m1.mesh.sizing.set_global_size(100)
m1.mesh.generation.generate(dim=3)

fem = m1.mesh.queries.get_fem_data(remove_orphans=True)

m1.end()

## OpenSees model (same as v4)

- `ndf = 6` global, solid column nodes overridden to `ndf = 3`.
- Grounded helper node at `GROUND_TAG` fixed in 6 DOFs, connected to the base reference point via a stiff 6-DOF `zeroLength` spring.
- `constraints('Penalty')` because the phantom nodes are daisy-chained.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

ops.timeSeries('Linear', 1)

# --- Nodes -----------------------------------------------------------------
for nid, xyz in fem.nodes.get(target='pg_column'):
    ops.node(nid, *xyz, '-ndf', 3)

ref = fem.nodes.get(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

# --- Grounded helper node + zeroLength spring at the base ------------------
base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

GROUND_TAG = 10_000
ops.node(GROUND_TAG, *base_xyz)
ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

K_SPRING = 1e14
for i in range(6):
    ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

ZL_TAG = 20_000
ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
            '-mat', 100, 101, 102, 103, 104, 105,
            '-dir', 1, 2, 3, 4, 5, 6)

# --- Material + tet elements -----------------------------------------------
E  = 200e3
nu = 0.3
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

# --- MP constraints --------------------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

## Step-wise analysis with displacement snapshots

After each `ops.analyze(1)` call we query `ops.nodeDisp` for every node in `fem.nodes.ids` and append a step dict. The pseudo-time stored in each step is the current load factor (`ops.getTime()`), so the slider labels are meaningful.

In [ ]:
node_ids_list = [int(nid) for nid in fem.nodes.ids]
n_nodes = len(node_ids_list)
n_steps = 10

steps: list[dict] = []

for i in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'Step {i+1}: failed ({ok})')
        break

    t = float(ops.getTime())

    disp = np.zeros((n_nodes, 3), dtype=np.float64)
    for j, nid in enumerate(node_ids_list):
        d = ops.nodeDisp(nid)
        disp[j, 0] = d[0]
        disp[j, 1] = d[1]
        disp[j, 2] = d[2]

    u_mag = np.linalg.norm(disp, axis=1)

    steps.append({
        'time': t,
        'point_data': {
            'Displacement': disp,
            '|U|':          u_mag,
            'Ux':           disp[:, 0].copy(),
            'Uy':           disp[:, 1].copy(),
            'Uz':           disp[:, 2].copy(),
        },
    })

    print(f'Step {i+1:2d}  λ = {t:.2f}   |U|_max = {u_mag.max():.4e}')

print(f'\nCaptured {len(steps)} steps.')

## Base reaction at the final step

Expected from equilibrium: `Fy = -1000 N`, `Mx = +2·10⁶ N·mm`.

In [ ]:
ops.reactions()

rxn = ops.nodeReaction(GROUND_TAG)

print(f'nodeReaction(ground = {GROUND_TAG}):')
print(f'  Fx = {rxn[0]:+.4e}   Fy = {rxn[1]:+.4e}   Fz = {rxn[2]:+.4e}')
print(f'  Mx = {rxn[3]:+.4e}   My = {rxn[4]:+.4e}   Mz = {rxn[5]:+.4e}')
print(f'\nExpected:  Fy = -1.0000e+03   Mx = +2.0000e+06')

## Build the time-series Results and launch the viewer

Passing `steps=[...]` instead of `point_data=...` produces a time-series result. The viewer writes a PVD + 10 VTU files to a temp directory and opens them together; the time-step slider in the controls panel is automatically enabled and labelled with the load factor.

Drag the slider to scrub through the loading history. Combine with the Deformation toggle to see the column progressively bend into its final shape.

In [ ]:
results = Results.from_fem(
    fem,
    steps=steps,
    name='column_nts_v5',
)

print(f'Results: {len(steps)} steps, {n_nodes} nodes')
print(f'Time range: [{steps[0]["time"]:.2f}, {steps[-1]["time"]:.2f}]')
results

In [ ]:
results.viewer(blocking=False)

## Cantilever closed-form sanity check (final step)

Strong-axis bending of the W section (`Ix = bf·H³/12 − (bf − tw)·h³/12`, `H = 340 mm`):

- `δ_EB = P·L³ / (3·E·Ix)`  (Euler-Bernoulli)
- `δ_T  = δ_EB + P·L / (G·As)`, `As ≈ tw·h`  (Timoshenko, adds shear)

In [ ]:
top_id = int(next(iter(fem.nodes.get_ids(target='top'))))
tip_disp = ops.nodeDisp(top_id)

bf, tf, h_web, tw, L = 150.0, 20.0, 300.0, 10.0, 2000.0
H  = 2 * tf + h_web
Ix = (bf * H**3) / 12 - ((bf - tw) * h_web**3) / 12
P  = 1000.0

delta_EB    = P * L**3 / (3 * E * Ix)
G           = E / (2 * (1 + nu))
As          = tw * h_web
delta_shear = P * L / (G * As)
delta_T     = delta_EB + delta_shear

print(f'Top node {top_id} displacement (final step):')
print(f'  ux = {tip_disp[0]:+.4e}   uy = {tip_disp[1]:+.4e}   uz = {tip_disp[2]:+.4e}')
print(f'\nCantilever closed-form:')
print(f'  delta_EB         = {delta_EB:.4e} mm')
print(f'  delta_Timoshenko = {delta_T:.4e} mm')
print(f'  FEM uy           = {tip_disp[1]:+.4e} mm')
print(f'  FEM / EB         = {tip_disp[1] / delta_EB:.4f}')
print(f'  FEM / Timoshenko = {tip_disp[1] / delta_T:.4f}')

## Tip displacement progression

Sanity check: since the problem is linear, `u_y(top)` should scale exactly linearly with the load factor. Plotting `|U|_max` across steps confirms this and cross-checks the stored time-series against the final tip displacement.

In [ ]:
lambdas = np.array([s['time'] for s in steps])
umax    = np.array([s['point_data']['|U|'].max() for s in steps])

print('  step    λ        |U|_max')
print('  ----  ------  -----------')
for i, (lam, u) in enumerate(zip(lambdas, umax), 1):
    print(f'  {i:>4}  {lam:6.2f}  {u:11.4e}')

ratio = umax[-1] / lambdas[-1] if lambdas[-1] else 0
residuals = np.abs(umax - ratio * lambdas)
print(f'\nLinearity check:')
print(f'  |U|_max / λ (const):   {ratio:.4e}')
print(f'  max abs residual:      {residuals.max():.3e}')
print(f'  (a linear system should give residual ~ 0)')